In [53]:
from itertools import chain
from pathlib import Path

import dedupe
import polars as pl
from sklearn.metrics import precision_recall_curve

In [2]:
DATASET_PATH = Path("datasets/features_dataset_20260706.parquet")

# Chargement du jeu de données

In [3]:
df_features = pl.read_parquet(DATASET_PATH)

In [4]:
df_features.shape

(2000, 84)

In [5]:
df_train = df_features.filter(pl.col("split") == "train")
df_test = df_features.filter(pl.col("split") == "test")
print(df_train.shape, df_test.shape)

(1600, 84) (400, 84)


In [6]:
df_train.describe()

statistic,identifiant_unique_i,identifiant_unique_j,label,nom_i,description_i,acteur_type_id_i,adresse_i,adresse_complement_i,code_postal_i,ville_i,url_i,email_i,telephone_i,nom_commercial_i,nom_officiel_i,siren_i,siret_i,source_id_i,naf_principal_i,horaires_osm_i,horaires_description_i,public_accueilli_i,reprise_i,exclusivite_de_reprisereparation_i,uniquement_sur_rdv_i,consignes_dacces_i,action_principale_id_i,lieu_prestation_i,latitude_i,longitude_i,code_commune_insee_i,epci_id_i,action_reparer_i,action_acheter_i,action_revendre_i,action_donner_i,…,adresse_complement_j,code_postal_j,ville_j,url_j,email_j,telephone_j,nom_commercial_j,nom_officiel_j,siren_j,siret_j,source_id_j,naf_principal_j,horaires_osm_j,horaires_description_j,public_accueilli_j,reprise_j,exclusivite_de_reprisereparation_j,uniquement_sur_rdv_j,consignes_dacces_j,action_principale_id_j,lieu_prestation_j,latitude_j,longitude_j,code_commune_insee_j,epci_id_j,action_reparer_j,action_acheter_j,action_revendre_j,action_donner_j,action_louer_j,action_mettreenlocation_j,action_emprunter_j,action_preter_j,action_echanger_j,action_trier_j,action_rapporter_j,split
str,str,str,f64,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,…,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
"""count""","""1600""","""1600""",1600.0,"""1600""","""65""",1600.0,"""1579""","""147""","""1589""","""1599""","""238""","""142""","""458""","""436""","""7""","""805""","""701""",1278.0,"""123""","""8""","""41""","""1212""","""266""",276.0,314.0,"""209""",14.0,"""463""",1600.0,1600.0,"""1563""",1562.0,1576.0,1576.0,1576.0,1576.0,…,"""103""","""1589""","""1588""","""218""","""179""","""447""","""401""","""6""","""726""","""714""",1538.0,"""216""","""6""","""45""","""1126""","""193""",273.0,273.0,"""221""",12.0,"""367""",1600.0,1600.0,"""1551""",1550.0,1593.0,1593.0,1593.0,1593.0,1593.0,1593.0,1593.0,1593.0,1593.0,1593.0,1593.0,"""1600"""
"""null_count""","""0""","""0""",0.0,"""0""","""1535""",0.0,"""21""","""1453""","""11""","""1""","""1362""","""1458""","""1142""","""1164""","""1593""","""795""","""899""",322.0,"""1477""","""1592""","""1559""","""388""","""1334""",1324.0,1286.0,"""1391""",1586.0,"""1137""",0.0,0.0,"""37""",38.0,24.0,24.0,24.0,24.0,…,"""1497""","""11""","""12""","""1382""","""1421""","""1153""","""1199""","""1594""","""874""","""886""",62.0,"""1384""","""1594""","""1555""","""474""","""1407""",1327.0,1327.0,"""1379""",1588.0,"""1233""",0.0,0.0,"""49""",50.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,"""0"""
"""mean""",null,null,0.505,null,null,6.003125,null,null,null,null,null,null,null,null,null,null,null,112.028169,null,null,null,null,null,0.0,0.003185,null,10.5,null,2.370897,45.778403,null,574.631242,0.109772,0.022843,0.005076,0.074873,…,null,null,null,null,null,null,null,null,null,null,119.286736,null,null,null,null,null,0.003663,0.025641,null,10.416667,null,2.435897,45.82513,null,558.834839,0.173258,0.031387,0.006277,0.067797,0.008788,0.0,0.022599,0.0,0.009416,0.766478,0.010672,null
"""std""",null,null,null,null,null,2.838248,null,null,null,null,null,null,null,null,null,null,null,104.500351,null,null,null,null,null,null,null,null,1.870829,null,6.874362,5.339145,null,358.151183,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,107.554191,null,null,null,null,null,null,null,null,2.020726,null,6.63506,5.484027,null,354.706582,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""016256a6-f4be-5bf6-867a-487a54…","""1e0bfede-8c6b-5469-9b92-b458bf…",0.0,"""2 A Sports""","""Atelier bois + ressourcerie""",1.0,"""/""","""(sortie Carrefour)""","""01000""","""ALLEX""","""http://www.ag2iweb.com""","""BAYONNE.sav@leroymerlin.fr""","""+33240963172""","""ALDIS""","""AMBIANCES DU PASSE""","""200000586""","""200004802""",1.0,"""13.92Z""","""Mo 10:00-13:00,14:30-19:00; Tu…

# Transformation du jeu de données au format dedupe

In [7]:
FEATURES_NAMES_FROM_DATASET = [
    "nom",
    "description",
    "acteur_type_id",
    "adresse",
    "adresse_complement",
    "code_postal",
    "ville",
    "url",
    "email",
    "telephone",
    "nom_commercial",
    "nom_officiel",
    "siren",
    "siret",
    "naf_principal",
    "horaires_osm",
    "horaires_description",
    "horaires_osm",
    "public_accueilli",
    "reprise",
    "exclusivite_de_reprisereparation",
    "reprise",
    "uniquement_sur_rdv",
    "consignes_dacces",
    "action_principale_id",
    "lieu_prestation",
    "code_commune_insee",
    "epci_id",
    "action_reparer",
    "action_acheter",
    "action_revendre",
    "action_donner",
    "action_louer",
    "action_mettreenlocation",
    "action_emprunter",
    "action_preter",
    "action_echanger",
    "action_trier",
    "action_rapporter",
]

In [8]:
def transform_dataset_to_dedupe_format(
    df_features: pl.DataFrame,
) -> tuple[list[tuple[dict]]]:

    match_pairs = []
    distinct_pairs = []

    for row in df_features.iter_rows(named=True):
        actors = []
        for suffix in ["_i", "_j"]:
            actor_formatted = {}
            for feature_name in FEATURES_NAMES_FROM_DATASET:
                value = row[f"{feature_name}{suffix}"]
                if isinstance(value, int) or isinstance(value, float):
                    value = str(value)
                actor_formatted[feature_name] = value

            actor_formatted["location"] = (
                row[f"latitude{suffix}"],
                row[f"longitude{suffix}"],
            )
            actor_formatted["identifiant_unique"] = row[f"identifiant_unique{suffix}"]
            actors.append(actor_formatted)

        actors = tuple(actors)
        if row["label"]:
            match_pairs.append(actors)
        else:
            distinct_pairs.append(actors)

    return match_pairs, distinct_pairs

In [9]:
match_pairs, distinct_pairs = transform_dataset_to_dedupe_format(df_train)

# Configuration du modèle

In [10]:
variables = [
    dedupe.variables.String(name="nom", field="nom"),
    dedupe.variables.Text("description", has_missing=True),
    dedupe.variables.Categorical(
        "acteur_type_id",
        categories=["1", "2", "3", "4", "5", "7", "8", "9", "10", "11", "12"],
    ),
    dedupe.variables.String(name="adresse", field="adresse", has_missing=True),
    dedupe.variables.String(
        name="adresse_complement", field="adresse_complement", has_missing=True
    ),
    dedupe.variables.Exact(name="code_postal", field="code_postal", has_missing=True),
    dedupe.variables.String(name="ville", field="ville", has_missing=True),
    dedupe.variables.Exact("url", has_missing=True),
    dedupe.variables.Exact("email", has_missing=True),
    dedupe.variables.Exact("telephone", has_missing=True),
    dedupe.variables.String(
        name="nom_commercial", field="nom_commercial", has_missing=True
    ),
    dedupe.variables.String(
        name="nom_officiel", field="nom_officiel", has_missing=True
    ),
    dedupe.variables.Exact("siren", has_missing=True),
    dedupe.variables.Exact("siret", has_missing=True),
    dedupe.variables.Exact("naf_principal", has_missing=True),
    dedupe.variables.Text(name="horaires_osm", field="horaires_osm", has_missing=True),
    dedupe.variables.Text(
        name="horaires_description", field="horaires_description", has_missing=True
    ),
    dedupe.variables.Categorical(
        "public_accueilli",
        categories=[
            "Aucun",
            "Particuliers",
            "Particuliers et professionnels",
            "Professionnels",
        ],
        has_missing=True,
    ),
    dedupe.variables.Categorical(
        name="reprise",
        field="reprise",
        categories=["1 pour 0", "1 pour 1"],
        has_missing=True,
    ),
    dedupe.variables.Categorical(
        name="exclusivite_de_reprisereparation",
        field="exclusivite_de_reprisereparation",
        categories=["True", "False"],
        has_missing=True,
    ),
    dedupe.variables.Categorical(
        "uniquement_sur_rdv", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Text("consignes_dacces", has_missing=True),
    dedupe.variables.Categorical(
        "action_principale_id",
        categories=["7", "8", "5", "6", "1", "9", "3", "2", "12", "11", "4"],
        has_missing=True,
    ),
    dedupe.variables.Categorical(
        "lieu_prestation",
        categories=["A_DOMICILE", "SUR_PLACE", "SUR_PLACE_OU_A_DOMICILE"],
        has_missing=True,
    ),
    dedupe.variables.LatLong("location"),
    dedupe.variables.Exact("code_commune_insee", has_missing=True),
    dedupe.variables.Exact("epci_id", has_missing=True),
    dedupe.variables.Categorical(
        "action_reparer", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_acheter", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_revendre", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_donner", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_louer", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_mettreenlocation", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_emprunter", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_preter", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_echanger", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_trier", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Categorical(
        "action_rapporter", categories=["True", "False"], has_missing=True
    ),
    dedupe.variables.Interaction(
        "nom",
        "nom_commercial",
        "nom_officiel",
    ),
    dedupe.variables.Interaction(
        "adresse",
        "adresse_complement",
        "code_postal",
        "ville",
    ),
    dedupe.variables.Interaction("horaires_osm", "horaires_description"),
    dedupe.variables.Interaction("reprise", "exclusivite_de_reprisereparation"),
]

# Entrainement

In [11]:
model = dedupe.Dedupe(variable_definition=variables)

In [12]:
training = {
    "match": match_pairs,
    "distinct": distinct_pairs,
}

with (DATASET_PATH.parent / "training.json").open("w") as f:
    dedupe.write_training(training, f)

In [15]:
train_dataset = {}
for pairs in chain(match_pairs, distinct_pairs):
    for pair in pairs:
        train_dataset[pair["identifiant_unique"]] = pair

In [57]:
if (model_path:=Path("model.bin")).exists():
    with model_path.open("rb") as f:
        model = dedupe.StaticDedupe(f)
else:
    with (DATASET_PATH.parent / "training.json").open("r") as f:
        model.prepare_training(data=train_dataset, training_file=f)
    model.train()
    with Path("model.bin").open("wb") as f:
        model.write_settings(f)

In [33]:
pairs_to_score = []
for pairs in chain(match_pairs, distinct_pairs):
    tmp = []
    for pair in pairs:
        tmp.append((pair["identifiant_unique"], pair))
    pairs_to_score.append(tuple(tmp))

In [58]:
scores = model.score(pairs_to_score)

In [48]:
scores_list = []
for score in scores:
    scores_list.append(
        {
            "identifiant_unique_i": score[0][0],
            "identifiant_unique_j": score[0][1],
            "score": score[1],
        }
    )

In [49]:
df_scores = pl.DataFrame(scores_list)

In [52]:
df_train_with_scores = df_train.join(
    df_scores,
    on=["identifiant_unique_i", "identifiant_unique_j"],
    how="inner",
    validate="1:1",
)

# Evaluation sur train

In [55]:
df_train_with_scores.select(pl.col("label").astype(pl.Float))

AttributeError: 'Expr' object has no attribute 'astype'

In [ ]:
precision_recall_curve(df_train_with_scores.select(pl.col("label"), y_score, *, pos_label=None, sample_weight=None, drop_intermediate=False)